# HU-6 · Carga en MySQL y vista de KPIs

**Objetivo:** cargar el CSV final de la HU-5 (limpio + cluster) en MySQL y crear una vista con KPIs por medicamento que sirva de fuente a Power BI.

**Flujo (igual que el lab *Load_csv_parquet_to_MySQL*):**
1. Cargar el CSV de la HU-5
2. Recuperar del Excel de la HU-4 las columnas que necesitan los KPIs y no están en el CSV
3. Elegir las columnas que van a MySQL
4. Conectar con MySQL
5. Crear la tabla *(lo escribes tú)*
6. Insertar las filas y hacer commit
7. Verificar en SQL
8. Crear la vista de KPIs
9. Conclusiones

In [25]:
import json
import os
from pathlib import Path

import pandas as pd
import numpy as np
import mysql.connector

RAIZ = Path.cwd().resolve()
if RAIZ.name == "codigo_HU4_HU5":
    RAIZ = RAIZ.parent

RUTA_CLUSTERS = RAIZ / "datos" / "medicamentos_artritis_reumatoide_clusters.csv"
RUTA_HU4 = RAIZ / "datos" / "medicamentos_artritis_reumatoide_hu4.xlsx"

## 1. Cargar el CSV de la HU-5

In [2]:
df_clu = pd.read_csv(RUTA_CLUSTERS, dtype={"nregistro": str})
print(df_clu.shape)
print(df_clu.columns.tolist())

(645, 34)
['nregistro', 'nombre', 'volumen_informativo_seguridad', 'complejidad_desglose_clinico', 'indicador_riesgo_severo', 'pvp_iva', 'dosis_mg', 'antiguedad_anios', 'num_principios_activos', 'comercializado', 'generico', 'biosimilar', 'afecta_conduccion', 'triangulo_negro', 'tratamiento_larga_duracion', 'tiene_precio', 'tiene_revision', 'inyectable', 'estado_alta', 'estado_baja', 'via_intraarticular', 'via_intramuscular', 'via_intravenosa', 'via_oral', 'via_rectal', 'via_subcutanea', 'via_otras', 'forma_inyectable', 'forma_oral_liquido', 'forma_oral_solido', 'forma_perfusion', 'forma_rectal', 'cluster', 'nombre_cluster']


## 2. Recuperar las columnas que faltan
El CSV de la HU-5 no trae `cn`, `pactivos`, `precio_referencia`, `num_excipientes` ni `forma_farmaceutica_simplificada`, porque no se utilizaron para la creación de los clusters en HU-5. Sí están en el archivo de la HU-4.

In [3]:
# Importar nregistro como string para no perder los ceros que tienen a la izquierda algunos registros
df_hu4 = pd.read_excel(RUTA_HU4, dtype={"nregistro": str, "cn": str})

cols_extra = ["cn", "pactivos", "precio_referencia", "num_excipientes", "forma_farmaceutica_simplificada",
              "precio_rellenado"]
cols_faltan = [c for c in cols_extra if c not in df_clu.columns]
print("Se recuperan:", cols_faltan)

df = df_clu.merge(df_hu4[["nregistro"] + cols_faltan], on="nregistro", how="left", validate="1:1")

print(df_clu.shape, "->", df.shape)
print(df[cols_extra].isna().sum())
print((df["pvp_iva"] == df["precio_referencia"]).sum(), "filas con PVP == PR")

Se recuperan: ['cn', 'pactivos', 'precio_referencia', 'num_excipientes', 'forma_farmaceutica_simplificada', 'precio_rellenado']
(645, 34) -> (645, 40)
cn                                   0
pactivos                             0
precio_referencia                  329
num_excipientes                      0
forma_farmaceutica_simplificada      0
precio_rellenado                     0
dtype: int64
316 filas con PVP == PR


### 2.1 Quitar los precios inducidos
Durante el EDA, se rellenó el precio de 150 medicamentos que no están en el Nomenclátor (por ello se creó una variable que indique si el o¡precio es original o inducido, `precio_rellenado = 1`). Para el K-Means era una decisión válida, pero un KPI de negocio no puede calcularse con precios que no sean correctos al 100%.

In [4]:
print("Precios rellenados:", (df["precio_rellenado"] == 1).sum())
df.loc[df["precio_rellenado"] == 1, "pvp_iva"] = None
df["tiene_precio"] = df["pvp_iva"].notna().astype(int)

print("pvp_iva nulos ahora:", df["pvp_iva"].isna().sum())   # deben ser 306 = SIN DATO
print(pd.crosstab(df_hu4.set_index("nregistro").loc[df["nregistro"], "estado_nomenclator"].values,
                  df["pvp_iva"].isna(), rownames=["estado_nomenclator"], colnames=["pvp nulo"]))

Precios rellenados: 150
pvp_iva nulos ahora: 306
pvp nulo            False  True 
estado_nomenclator              
ALTA                  247      0
BAJA                   92      0
SIN DATO                0    306


## 3. Elegir las columnas para MySQL
Solo las necesarias para calcular los KPIs y para filtrar en Power BI. Las columnas 0/1 de vías y formas repiten lo que ya dice `forma_farmaceutica_simplificada`.

In [5]:
columnas = ["nregistro", "cn", "nombre", "pactivos", "forma_farmaceutica_simplificada", "dosis_mg",
            "cluster", "nombre_cluster", "pvp_iva", "precio_referencia", "tiene_precio",
            "volumen_informativo_seguridad", "complejidad_desglose_clinico", "indicador_riesgo_severo",
            "num_principios_activos", "num_excipientes", "comercializado", "generico", "biosimilar",
            "antiguedad_anios"]
df_load = df[columnas].copy()

print(df_load.shape)
print(df_load.isna().sum()[lambda s: s > 0])
print("nregistro único:", df_load["nregistro"].is_unique, "| cn único:", df_load["cn"].is_unique)

(645, 20)
dosis_mg               5
pvp_iva              306
precio_referencia    329
dtype: int64
nregistro único: True | cn único: True


## 4. Conectar con MySQL
La conexión usa el `config.json` local de la raíz del repositorio, excluido de Git. Si el campo `password` está vacío o no existe, se usa la variable de entorno `ARTHENA_MYSQL_PWD`.

In [ ]:
with (RAIZ / "config.json").open(encoding="utf-8") as archivo:
    config = json.load(archivo)

clave = config.get("password") or os.getenv("ARTHENA_MYSQL_PWD")
if not clave:
    raise RuntimeError("Falta la clave de MySQL en config.json o en ARTHENA_MYSQL_PWD")

conn = mysql.connector.connect(
    host=config.get("host", "localhost"),
    port=int(config.get("port", 3306)),
    user=config.get("user", "arthena_app"),
    password=clave,
    database=config.get("database", "arthena"),
)
print("Conectado:", conn.is_connected())

## 5. Crear la tabla

In [7]:
print(df_load.dtypes)
for col in ["nregistro", "cn", "nombre", "pactivos", "forma_farmaceutica_simplificada", "nombre_cluster"]:
    print(col, "longitud máx:", df_load[col].astype(str).str.len().max())

nregistro                           object
cn                                  object
nombre                              object
pactivos                            object
forma_farmaceutica_simplificada     object
dosis_mg                           float64
cluster                              int64
nombre_cluster                      object
pvp_iva                            float64
precio_referencia                  float64
tiene_precio                         int64
volumen_informativo_seguridad        int64
complejidad_desglose_clinico         int64
indicador_riesgo_severo              int64
num_principios_activos               int64
num_excipientes                      int64
comercializado                       int64
generico                             int64
biosimilar                           int64
antiguedad_anios                   float64
dtype: object
nregistro longitud máx: 10
cn longitud máx: 6
nombre longitud máx: 93
pactivos longitud máx: 68
forma_farmaceutica_simplificad

In [15]:
# CREATE TABLE
sql_tabla = """
CREATE TABLE medicamentos (
    nregistro                        VARCHAR(10)   NOT NULL,
    cn                               VARCHAR(10)   NOT NULL,
    nombre                           VARCHAR(300)  NOT NULL,
    pactivos                         VARCHAR(500),
    forma_farmaceutica_simplificada  VARCHAR(100),
    dosis_mg                         DECIMAL(10,2),
    cluster                          TINYINT       NOT NULL,
    nombre_cluster                   VARCHAR(100)  NOT NULL,
    pvp_iva                          DECIMAL(10,2),
    precio_referencia                DECIMAL(10,2),
    tiene_precio                     TINYINT(1)    NOT NULL,
    volumen_informativo_seguridad    INT,
    complejidad_desglose_clinico     INT,
    indicador_riesgo_severo          INT,
    num_principios_activos           TINYINT,
    num_excipientes                  TINYINT,
    comercializado                   TINYINT(1),
    generico                         TINYINT(1),
    biosimilar                       TINYINT(1),
    antiguedad_anios                 DECIMAL(5,1),
    PRIMARY KEY (nregistro)
)"""
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS medicamentos")
cursor.execute(sql_tabla)
cursor.execute("DESCRIBE medicamentos")
for fila in cursor.fetchall(): print(fila)

('nregistro', 'varchar(10)', 'NO', 'PRI', None, '')
('cn', 'varchar(10)', 'NO', '', None, '')
('nombre', 'varchar(300)', 'NO', '', None, '')
('pactivos', 'varchar(500)', 'YES', '', None, '')
('forma_farmaceutica_simplificada', 'varchar(100)', 'YES', '', None, '')
('dosis_mg', 'decimal(10,2)', 'YES', '', None, '')
('cluster', 'tinyint', 'NO', '', None, '')
('nombre_cluster', 'varchar(100)', 'NO', '', None, '')
('pvp_iva', 'decimal(10,2)', 'YES', '', None, '')
('precio_referencia', 'decimal(10,2)', 'YES', '', None, '')
('tiene_precio', 'tinyint(1)', 'NO', '', None, '')
('volumen_informativo_seguridad', 'int', 'YES', '', None, '')
('complejidad_desglose_clinico', 'int', 'YES', '', None, '')
('indicador_riesgo_severo', 'int', 'YES', '', None, '')
('num_principios_activos', 'tinyint', 'YES', '', None, '')
('num_excipientes', 'tinyint', 'YES', '', None, '')
('comercializado', 'tinyint(1)', 'YES', '', None, '')
('generico', 'tinyint(1)', 'YES', '', None, '')
('biosimilar', 'tinyint(1)', 'YES'

## 6. Insertar las filas y hacer commit

In [16]:
df_sql = df_load.astype(object).where(df_load.notna(), None)
filas = list(df_sql.itertuples(index=False, name=None))

marcadores = ", ".join(["%s"] * len(columnas))
sql_insert = f"INSERT INTO medicamentos ({', '.join(columnas)}) VALUES ({marcadores})"

cursor.execute("DELETE FROM medicamentos")
cursor.executemany(sql_insert, filas)

cursor.execute("SELECT COUNT(*) FROM medicamentos")
n_mysql = cursor.fetchone()[0]
print("Filas en pandas:", len(df_load), "| filas en MySQL:", n_mysql)

if n_mysql == len(df_load):
    conn.commit(); print("COMMIT hecho")
else:
    conn.rollback(); print("Los conteos no cuadran: ROLLBACK")

Filas en pandas: 645 | filas en MySQL: 645
COMMIT hecho


## 7. Verificar en SQL
Comprobaciones directamente contra MySQL: que la tabla tiene las mismas filas, nulos y valores que el DataFrame utilizado para poblarla.

Para las consultas de lectura se usa `pd.read_sql`, que devuelve el resultado como DataFrame.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# Comprobar el número de registros
pd.read_sql("""
    SELECT COUNT(*)                  AS filas,
           COUNT(DISTINCT nregistro) AS nregistro_distintos,
           COUNT(DISTINCT cn)        AS cn_distintos
    FROM medicamentos
""", conn)

,filas,nregistro_distintos,cn_distintos
0,645,645,645


In [11]:
# Comprobar los nulos de la tabla
nulos_sql = pd.read_sql("""
    SELECT SUM(dosis_mg IS NULL)          AS dosis_mg,
           SUM(pvp_iva IS NULL)           AS pvp_iva,
           SUM(precio_referencia IS NULL) AS precio_referencia
    FROM medicamentos
""", conn).T.rename(columns={0: "nulos_mysql"})

nulos_sql["nulos_pandas"] = df_load[nulos_sql.index].isna().sum()
nulos_sql

,nulos_mysql,nulos_pandas
dosis_mg,5.0,5
pvp_iva,306.0,306
precio_referencia,329.0,329


In [ ]:
# Comprobar que los registros con 0 a la izquierda no lo han pérdido
pd.read_sql("""
    SELECT nregistro, nombre
    FROM medicamentos
    WHERE nregistro LIKE '0%'
""", conn)

,nregistro,nombre
0,03256003,HUMIRA 40 mg SOLUCION INYECTABLE EN JERINGA PR...
1,03256008,HUMIRA 40 mg SOLUCION INYECTABLE EN PLUMA PREC...
2,07389001,ORENCIA 250 mg POLVO PARA CONCENTRADO PARA SOL...
3,07389008,ORENCIA 125 mg SOLUCION INYECTABLE EN JERINGA ...
4,08492001,"ROACTEMRA 20 mg/ml, CONCENTRADO PARA SOLUCION ..."
5,08492003,"ROACTEMRA 20 mg/ml, CONCENTRADO PARA SOLUCION ..."
6,09546001,SIMPONI 50 MG SOLUCION INYECTABLE EN PLUMA PRE...
7,09604001,LEFLUNOMIDE ZENTIVA 10 mg COMPRIMIDOS RECUBIER...
8,09604005,LEFLUNOMIDE ZENTIVA 20 mg COMPRIMIDOS RECUBIER...
9,09604010,LEFLUNOMIDE ZENTIVA 100 mg COMPRIMIDOS RECUBIE...


In [ ]:
# Comprobar la distribución de los clusters
clusters_sql = pd.read_sql("""
    SELECT cluster, nombre_cluster, COUNT(*) AS n
    FROM medicamentos
    GROUP BY cluster, nombre_cluster
    ORDER BY cluster
""", conn)
clusters_sql["n_pandas"] = clusters_sql["cluster"].map(df_load["cluster"].value_counts())
clusters_sql

,cluster,nombre_cluster,n,n_pandas
0,0,Coxibs (antiinflamatorios selectivos),159,159
1,1,AINE genéricos económicos,185,185
2,2,Biológicos y terapias dirigidas,101,101
3,3,Clásicos de larga trayectoria,56,56
4,4,Fichas breves (corticoides y analgésicos),26,26
5,5,FAME e inmunosupresores,118,118


## 8. Vista de KPIs
### 8.1 Comprobación de cada KPI

Antes de crear la vista se prueba cada KPI por separado para comprobar que el resultado tiene sentido.

### **Desviación de Precio (%)**
(PVP-IVA − PR) / PR × 100. `NULLIF(precio_referencia, 0)` evita dividir entre cero: si el PR es 0, el resultado es `NULL` en lugar de un error.

        (pvp_iva - precio_referencia) / NULLIF(precio_referencia, 0) * 100

In [15]:
pd.read_sql("""
    SELECT COUNT(*)                                                         AS total,
           COUNT((pvp_iva - precio_referencia) / NULLIF(precio_referencia, 0) * 100) AS con_valor,
           SUM((pvp_iva - precio_referencia) / NULLIF(precio_referencia, 0) * 100 = 0) AS igual_a_0,
           MIN((pvp_iva - precio_referencia) / NULLIF(precio_referencia, 0) * 100) AS minimo,
           MAX((pvp_iva - precio_referencia) / NULLIF(precio_referencia, 0) * 100) AS maximo
    FROM medicamentos
""", conn)

,total,con_valor,igual_a_0,minimo,maximo
0,645,316,316.0,0.0,0.0


En todos los medicamentos con precio de referencia la desviación es 0 %: el PVP coincide con el PR (ya se vio en el EDA de la HU-4). El resto sale `NULL` porque no tienen PR.

### **Margen Económico Absoluto**
PVP-IVA − PR. Por la misma razón, debe salir 0 € donde existen los dos precios.

        pvp_iva - precio_referencia

In [16]:
pd.read_sql("""
    SELECT COUNT(pvp_iva - precio_referencia)      AS con_valor,
           SUM(pvp_iva - precio_referencia = 0)    AS igual_a_0,
           MIN(pvp_iva - precio_referencia)        AS minimo,
           MAX(pvp_iva - precio_referencia)        AS maximo
    FROM medicamentos
""", conn)

,con_valor,igual_a_0,minimo,maximo
0,316,316.0,0.0,0.0


### **Densidad de la Ficha Técnica** 
palabras de la sección 4.4 / número de tablas. Las fichas sin ninguna tabla dan `NULL` gracias a `NULLIF`.

        volumen_informativo_seguridad / NULLIF(complejidad_desglose_clinico, 0)

In [17]:
pd.read_sql("""
    SELECT COUNT(*)                                                                   AS total,
           SUM(complejidad_desglose_clinico = 0)                                      AS fichas_sin_tablas,
           COUNT(volumen_informativo_seguridad / NULLIF(complejidad_desglose_clinico, 0)) AS con_valor,
           MIN(volumen_informativo_seguridad / NULLIF(complejidad_desglose_clinico, 0))   AS minimo,
           AVG(volumen_informativo_seguridad / NULLIF(complejidad_desglose_clinico, 0))   AS media,
           MAX(volumen_informativo_seguridad / NULLIF(complejidad_desglose_clinico, 0))   AS maximo
    FROM medicamentos
""", conn)

,total,fichas_sin_tablas,con_valor,minimo,media,maximo
0,645,187.0,458,69.1053,764.25563,2711.0


In [18]:
# Las fichas más y menos densas
pd.read_sql("""
    SELECT nombre, volumen_informativo_seguridad, complejidad_desglose_clinico, densidad
    FROM (SELECT nombre, volumen_informativo_seguridad, complejidad_desglose_clinico,
                 volumen_informativo_seguridad / complejidad_desglose_clinico AS densidad,
                 ROW_NUMBER() OVER (ORDER BY volumen_informativo_seguridad / complejidad_desglose_clinico DESC) AS pos_mas,
                 ROW_NUMBER() OVER (ORDER BY volumen_informativo_seguridad / complejidad_desglose_clinico ASC)  AS pos_menos
          FROM medicamentos
          WHERE complejidad_desglose_clinico > 0) t
    WHERE pos_mas <= 3 OR pos_menos <= 3
    ORDER BY densidad DESC
""", conn)

,nombre,volumen_informativo_seguridad,complejidad_desglose_clinico,densidad
0,IMMUFALK 100 MG COMPRIMIDOS RECUBIERTOS CON PE...,2711,1,2711.0000
1,IMMUFALK 75 MG COMPRIMIDOS RECUBIERTOS CON PEL...,2711,1,2711.0000
2,CIQORIN 100 MG CAPSULAS BLANDAS EFG,2467,1,2467.0000
3,AMGEVITA 40 MG SOLUCION INYECTABLE EN PLUMA PR...,2624,35,74.9714
4,Humira 40mg solucion inyectable en jeringa pre...,2626,38,69.1053
5,HUMIRA 40 MG SOLUCIÓN INYECTABLE EN PLUMA PREC...,2626,38,69.1053


Las fichas menos densas son las de los biológicos (HUMIRA, AMGEVITA…): tienen muchas tablas que reparten el texto. Las más densas tienen mucho texto de advertencias y casi ninguna tabla.

### **Total de Componentes**
principios activos + excipientes. `COALESCE(..., 0)` trata un nulo como 0 para que la suma no salga `NULL`.

        SELECT COALESCE(num_principios_activos, 0) + COALESCE(num_excipientes, 0) AS total_componentes

In [ ]:
pd.read_sql("""
    SELECT COALESCE(num_principios_activos, 0) + COALESCE(num_excipientes, 0) AS total_componentes,
           COUNT(*)             AS n,
           MAX(num_excipientes) AS max_excipientes
    FROM medicamentos
    GROUP BY total_componentes
    ORDER BY total_componentes
""", conn)

,total_componentes,n,max_excipientes
0,1,636,0
1,2,8,0
2,3,1,0


`num_excipientes` vale 0 en todos los medicamentos (dato de la HU-1), así que el total de componentes es igual al número de principios activos.

### **Ranking de precios por forma farmacéutica.**
`RANK()` numera los medicamentos de cada forma farmacéutica del más caro (1) al más barato; los empates comparten posición.

MySQL ordena los `NULL` al final, pero **les asigna igualmente una posición**. En las formas donde ningún medicamento tiene precio (perfusiones, comprimidos efervescentes), todos salían como "puesto 1", es decir, como los más caros. Por eso en la vista el ranking se deja en `NULL` cuando no hay precio:

       RANK() OVER (PARTITION BY forma_farmaceutica_simplificada ORDER BY pvp_iva DESC) AS ranking_original,

           CASE WHEN pvp_iva IS NULL THEN NULL

                ELSE RANK() OVER (PARTITION BY forma_farmaceutica_simplificada ORDER BY pvp_iva DESC)

           END AS ranking_corregido

In [ ]:
pd.read_sql("""
    SELECT forma_farmaceutica_simplificada, nombre, pvp_iva,
           RANK() OVER (PARTITION BY forma_farmaceutica_simplificada ORDER BY pvp_iva DESC) AS ranking_original,
           CASE WHEN pvp_iva IS NULL THEN NULL
                ELSE RANK() OVER (PARTITION BY forma_farmaceutica_simplificada ORDER BY pvp_iva DESC)
           END AS ranking_corregido
    FROM medicamentos
    WHERE forma_farmaceutica_simplificada IN ('INYECTABLE PERFUSION', 'COMPRIMIDO EFERVESCENTE')
    LIMIT 6
""", conn)

,forma_farmaceutica_simplificada,nombre,pvp_iva,ranking_original,ranking_corregido
0,COMPRIMIDO EFERVESCENTE,DOLMEN 500 MG/250 MG/10 MG COMPRIMIDOS EFERVES...,None,1,None
1,COMPRIMIDO EFERVESCENTE,PARACETAMOL KERN PHARMA 1 g COMPRIMIDOS EFERVE...,None,1,None
2,INYECTABLE PERFUSION,ORENCIA 250 mg POLVO PARA CONCENTRADO PARA SOL...,None,1,None
3,INYECTABLE PERFUSION,REMSIMA 40 MG/ML CONCENTRADO PARA SOLUCION PAR...,None,1,None
4,INYECTABLE PERFUSION,INFLECTRA 100 MG POLVO PARA CONCENTRADO PARA S...,None,1,None
5,INYECTABLE PERFUSION,FLIXABI 100MG POLVO PARA CONCENTRADO PARA SOLU...,None,1,None


In [21]:
# El medicamento más caro de cada forma farmacéutica (ranking = 1)
pd.read_sql("""
    SELECT forma_farmaceutica_simplificada, nombre, pvp_iva
    FROM (SELECT forma_farmaceutica_simplificada, nombre, pvp_iva,
                 RANK() OVER (PARTITION BY forma_farmaceutica_simplificada ORDER BY pvp_iva DESC) AS ranking
          FROM medicamentos
          WHERE pvp_iva IS NOT NULL) t
    WHERE ranking = 1
    ORDER BY pvp_iva DESC
""", conn)

,forma_farmaceutica_simplificada,nombre,pvp_iva
0,COMPRIMIDO,ARAVA 20 mg COMPRIMIDOS RECUBIERTOS CON PELICULA,167.48
1,COMPRIMIDO,LEFLUNOMIDA NORMON 20 mg COMPRIMIDOS RECUBIERT...,167.48
2,SOLUCION/SUSPENSION ORAL,SANDIMMUN NEORAL 100 mg/ml SOLUCION ORAL,87.42
3,CAPSULA,CIQORIN 100 MG CAPSULAS BLANDAS EFG,52.45
4,CAPSULA,SANDIMMUN NEORAL 100 mg CAPSULAS BLANDAS,52.45
5,INYECTABLE,"METHOFILL 30 MG/ 0,60 ML SOLUCION INYECTABLE E...",32.66
6,INYECTABLE,"METOJECT 30 mg/0,60 ml SOLUCION INYECTABLE EN ...",32.66
7,COMPRIMIDO LIBERACION MODIFICADA,VIMOVO 500 mg/20 mg COMPRIMIDOS DE LIBERACION ...,15.38
8,COMPRIMIDO LIBERACION MODIFICADA,NAPROXENO/ESOMEPRAZOL SANDOZ 500 MG/20 MG COMP...,15.38
9,SOLUCION/SUSPENSION GOTAS ORALES,ESTILSONA 7 mg/ml GOTAS ORALES EN SUSPENSIÓN,4.04


### **Índice de precio relativo por forma**
(KPI adicional) = PVP / precio medio de su forma farmacéutica. Un 1 indica que cuesta lo mismo que la media de su forma; un 2, el doble.

Comprobación: la media del índice dentro de cada forma tiene que salir 1 (salvo por el redondeo).

In [22]:
pd.read_sql("""
    SELECT forma_farmaceutica_simplificada,
           COUNT(indice)        AS con_precio,
           ROUND(AVG(indice), 2) AS media_indice,
           MIN(indice)          AS minimo,
           MAX(indice)          AS maximo
    FROM (SELECT forma_farmaceutica_simplificada,
                 ROUND(pvp_iva / NULLIF(AVG(pvp_iva) OVER (PARTITION BY forma_farmaceutica_simplificada), 0), 2) AS indice
          FROM medicamentos) t
    GROUP BY forma_farmaceutica_simplificada
    ORDER BY con_precio DESC
""", conn)

,forma_farmaceutica_simplificada,con_precio,media_indice,minimo,maximo
0,COMPRIMIDO,228,1.0,0.02,18.08
1,INYECTABLE,45,1.0,0.10,1.54
2,CAPSULA,28,1.0,0.09,2.95
3,SOLUCION/SUSPENSION ORAL,18,1.0,0.29,10.18
4,COMPRIMIDO LIBERACION MODIFICADA,6,1.0,0.34,2.08
5,SUPOSITORIO,5,1.0,0.92,1.14
6,SOLUCION/SUSPENSION GOTAS ORALES,3,1.0,0.94,1.12
7,SOLUCION/SUSPENSION ORAL EFERVESCENTE,3,1.0,1.00,1.00
8,CAPSULA LIBERACION MODIFICADA,2,1.0,1.00,1.00
9,COMPRIMIDO BUCODISPERSABLE/LIOTAB,1,1.0,1.00,1.00


### 8.2 Crear vista para exportar resultados
La vista incluye todas las columnas de la tabla más los KPIs. Como todos los KPIs se calculan fila a fila, una sola vista sirve de fuente para Power BI sin tener que volver a cruzar los datos.

In [24]:
sql_vista = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos AS
SELECT
    m.*,

    -- Desviación de Precio (%): (PVP-IVA - PR) / PR * 100
    (m.pvp_iva - m.precio_referencia) / NULLIF(m.precio_referencia, 0) * 100 AS desviacion_precio_pct,

    -- Densidad de la Ficha Técnica: palabras de la sección 4.4 / número de tablas
    m.volumen_informativo_seguridad / NULLIF(m.complejidad_desglose_clinico, 0) AS densidad_ficha_tecnica,

    -- Total de Componentes: principios activos + excipientes
    COALESCE(m.num_principios_activos, 0) + COALESCE(m.num_excipientes, 0) AS total_componentes,

    -- Margen Económico Absoluto: PVP-IVA - PR
    m.pvp_iva - m.precio_referencia AS margen_economico_absoluto,

    -- (PLUS) Ranking de precios dentro de cada forma farmacéutica (1 = el más caro; NULL si no hay precio)
    CASE WHEN m.pvp_iva IS NULL THEN NULL
         ELSE RANK() OVER (PARTITION BY m.forma_farmaceutica_simplificada ORDER BY m.pvp_iva DESC)
    END AS ranking_precio_forma,

    -- Índice de precio relativo: PVP / precio medio de su forma farmacéutica
    ROUND(m.pvp_iva / NULLIF(AVG(m.pvp_iva) OVER (PARTITION BY m.forma_farmaceutica_simplificada), 0), 2)
        AS indice_precio_relativo_forma

FROM medicamentos m
"""
cursor = conn.cursor()
cursor.execute(sql_vista)

cursor.execute("SHOW FULL TABLES WHERE Table_type = 'VIEW'")
print(cursor.fetchall())

[('vista_kpi_medicamentos', 'VIEW')]


### 8.3 Verificación con pandas

Se recalculan los mismos KPIs en pandas a partir de `df_load` (los datos que se cargaron) y se comparan con lo que devuelve la vista. Si las fórmulas SQL son correctas, las diferencias deben ser 0.

In [26]:
df_kpis = pd.read_sql("SELECT * FROM vista_kpi_medicamentos", conn)

cols_kpi = ["desviacion_precio_pct", "densidad_ficha_tecnica", "total_componentes",
            "margen_economico_absoluto", "ranking_precio_forma", "indice_precio_relativo_forma"]
df_kpis[cols_kpi] = df_kpis[cols_kpi].apply(pd.to_numeric)

# Los mismos KPIs calculados en pandas
kpis_pd = pd.DataFrame({"nregistro": df_load["nregistro"]})
kpis_pd["desviacion_precio_pct"] = (df_load["pvp_iva"] - df_load["precio_referencia"]) / df_load["precio_referencia"].replace(0, np.nan) * 100
kpis_pd["densidad_ficha_tecnica"] = df_load["volumen_informativo_seguridad"] / df_load["complejidad_desglose_clinico"].replace(0, np.nan)
kpis_pd["total_componentes"] = df_load["num_principios_activos"].fillna(0) + df_load["num_excipientes"].fillna(0)
kpis_pd["margen_economico_absoluto"] = df_load["pvp_iva"] - df_load["precio_referencia"]
kpis_pd["ranking_precio_forma"] = df_load.groupby("forma_farmaceutica_simplificada")["pvp_iva"].rank(method="min", ascending=False)
kpis_pd["indice_precio_relativo_forma"] = (df_load["pvp_iva"] /
                                           df_load.groupby("forma_farmaceutica_simplificada")["pvp_iva"].transform("mean")).round(2)

comparacion = df_kpis[["nregistro"] + cols_kpi].merge(kpis_pd, on="nregistro", suffixes=("_sql", "_pandas"))

resumen = []
for col in cols_kpi:
    sql_col, pd_col = comparacion[f"{col}_sql"], comparacion[f"{col}_pandas"]
    resumen.append({"kpi": col,
                    "nulos_sql": int(sql_col.isna().sum()),
                    "nulos_pandas": int(pd_col.isna().sum()),
                    "max_diferencia": (sql_col - pd_col).abs().max()})
pd.DataFrame(resumen)

,kpi,nulos_sql,nulos_pandas,max_diferencia
0,desviacion_precio_pct,329,329,0.000000
1,densidad_ficha_tecnica,187,187,0.000048
2,total_componentes,0,0,0.000000
3,margen_economico_absoluto,329,329,0.000000
4,ranking_precio_forma,306,306,0.000000
5,indice_precio_relativo_forma,306,306,0.000000


Los nulos coinciden en todos los KPIs y las diferencias son 0 (o casi 0 debido a formatos de números).

### 8.4 Consulta completa y exportación

Se exporta el resultado completo de la vista (datos + KPIs) a CSV. Así queda constancia del resultado en el repositorio y Power BI puede usarlo aunque no tenga conexión a MySQL.

De cualquier forma, si se quisiera hacer con conexión, Power BI puede leer directamente la vista `vista_kpi_medicamentos`.

In [28]:
RUTA_KPIS = "../datos/medicamentos_artritis_reumatoide_kpis.csv"

print(df_kpis.shape)
df_kpis.to_csv(RUTA_KPIS, index=False, encoding="utf-8-sig")
print("Exportado:", RUTA_KPIS)

df_kpis[["nregistro", "nombre", "forma_farmaceutica_simplificada", "nombre_cluster", "pvp_iva"] + cols_kpi].head(5)

(645, 26)
Exportado: ../datos/medicamentos_artritis_reumatoide_kpis.csv


,nregistro,nombre,forma_farmaceutica_simplificada,nombre_cluster,pvp_iva,desviacion_precio_pct,densidad_ficha_tecnica,total_componentes,margen_economico_absoluto,ranking_precio_forma,indice_precio_relativo_forma
0,78331,CIQORIN 100 MG CAPSULAS BLANDAS EFG,CAPSULA,FAME e inmunosupresores,52.45,0.0,2467.0,1,0.0,1.0,2.95
1,60320,SANDIMMUN NEORAL 100 mg CAPSULAS BLANDAS,CAPSULA,FAME e inmunosupresores,52.45,0.0,1254.0,1,0.0,1.0,2.95
2,60318,SANDIMMUN NEORAL 50 mg CAPSULAS BLANDAS,CAPSULA,FAME e inmunosupresores,26.23,0.0,1254.0,1,0.0,3.0,1.48
3,78333,CIQORIN 50 MG CAPSULAS BLANDAS EFG,CAPSULA,FAME e inmunosupresores,26.23,0.0,2467.0,1,0.0,3.0,1.48
4,84390,CELECOXIB KERN PHARMA 200 MG CAPSULAS DURAS EFG,CAPSULA,Coxibs (antiinflamatorios selectivos),22.28,0.0,1091.0,1,0.0,5.0,1.25


In [29]:
# Cerrar la conexión
cursor.close()
conn.close()
print("Conexión cerrada:", not conn.is_connected())

Conexión cerrada: True


## 9. Conclusiones

- **Carga:** los 645 medicamentos están en la tabla `medicamentos` de MySQL, con `nregistro` como clave primaria y sin perder los ceros a la izquierda. Los nulos y el reparto por cluster coinciden con los del CSV de la HU-5.
- **Precios:** en la tabla solo hay precios oficiales del Nomenclátor. Los 150 precios rellenados para el K-Means se quitaron, porque un KPI de negocio no puede calcularse con precios estimados.
- **Vista:** `vista_kpi_medicamentos` reúne todas las columnas de la tabla y los 6 KPIs (los 5 del enunciado más el índice de precio relativo). Se comprobó cada KPI por separado y se verificó con pandas que la vista calcula exactamente lo mismo.

**Lectura de los KPIs**
- **Desviación de precio y margen absoluto:** son 0 en los 316 medicamentos con precio de referencia, porque los medicamentos financiados se ajustan a su precio de referencia. En el resto salen `NULL`, porque no tienen PR.
- **Densidad de la ficha técnica:** separa bien los biológicos, cuyas fichas reparten el texto en muchas tablas (menor densidad). Sale `NULL` en las 187 fichas sin ninguna tabla.
- **Total de componentes:** coincide con el número de principios activos, porque `num_excipientes` vale 0 en todos los medicamentos (dato de la HU-1).
- **Ranking por forma farmacéutica:** se corrigió para que los medicamentos sin precio no aparezcan como los más caros de su forma.
- **Índice de precio relativo:** identifica los medicamentos que cuestan mucho más que la media de su forma; por ejemplo, la leflunomida de 167,48 € cuesta unas 18 veces la media de los comprimidos.

**Siguiente paso:** conectar Power BI a la vista `vista_kpi_medicamentos` (o al CSV exportado) para construir los reportes.